# EDA Completo v3 - Dataset Sintético
## Análisis de Calidad con Patrón de Auditoria3

Análisis exploratorio del dataset sintético generado con validaciones similares al pipeline original:
- **Análisis de defectos inyectados**: Ground truth registry
- **Validación de dominios**: Duplicados y formato
- **Cruce flota-telemetría**: Cobertura de dispositivos
- **Análisis de transacciones**: Combustible y tarjetas
- **Detección de anomalías**: Defectos por tipo
- **Estadísticas de calidad**: Resumen ejecutivo

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

drive.mount('/content/drive')
print("✓ Google Drive montado")

In [ ]:
# Cargar dataset
dataset_dir = Path('/content/drive/MyDrive/Integrador/datasets/early_stage_v3')

# Cargar todos los CSVs
tables = {}
for csv_file in dataset_dir.glob('*.csv'):
    if 'ejecucion' not in csv_file.name:  # Skip execution log for now
        table_name = csv_file.stem
        tables[table_name] = pd.read_csv(csv_file)
        print(f"✓ {table_name:30s} {len(tables[table_name]):6d} filas × {len(tables[table_name].columns):2d} cols")

# Cargar manifest
manifest_path = dataset_dir / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"\n✓ Manifest cargado (seed: {manifest['seed']})")

print(f"\n📊 Total registros: {sum(len(df) for df in tables.values()):,}")

## 1. ANÁLISIS DE DEFECTOS INYECTADOS

In [ ]:
gt = tables.get('ground_truth', pd.DataFrame())

if len(gt) > 0:
    print(f"🔴 DEFECTOS INYECTADOS")
    print(f"\nTotal: {len(gt)} defectos\n")
    
    print("POR TIPO:")
    tipo_counts = gt['tipo'].value_counts().sort_values(ascending=False)
    for tipo, count in tipo_counts.items():
        print(f"  {tipo:30s} {count:3d}")
    
    print(f"\nPOR SEVERIDAD:")
    sev_counts = gt['severidad'].value_counts()
    for sev, count in sev_counts.items():
        print(f"  {sev:20s} {count:3d}")
    
    print(f"\nPOR ENTIDAD:")
    ent_counts = gt['entidad'].value_counts()
    for ent, count in ent_counts.items():
        print(f"  {ent:20s} {count:3d}")
else:
    print("⚠ No hay defectos inyectados")

In [ ]:
# Visualización de defectos
if len(gt) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Análisis de Defectos Inyectados', fontsize=14, fontweight='bold')
    
    # Por tipo
    tipo_counts.plot(kind='barh', ax=axes[0, 0], color='#FF6B6B')
    axes[0, 0].set_title('Defectos por Tipo', fontweight='bold')
    axes[0, 0].set_xlabel('Cantidad')
    
    # Por severidad
    colors = {'alta': '#FF6B6B', 'media': '#FFA500', 'baja': '#FFD700'}
    sev_counts.plot(kind='bar', ax=axes[0, 1],
                    color=[colors.get(s, '#999') for s in sev_counts.index])
    axes[0, 1].set_title('Defectos por Severidad', fontweight='bold')
    axes[0, 1].set_ylabel('Cantidad')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Pie por severidad
    axes[1, 0].pie(sev_counts.values, labels=sev_counts.index, autopct='%1.1f%%',
                   colors=[colors.get(s, '#999') for s in sev_counts.index])
    axes[1, 0].set_title('Distribución por Severidad', fontweight='bold')
    
    # Por entidad
    ent_counts.plot(kind='bar', ax=axes[1, 1], color='#4CAF50')
    axes[1, 1].set_title('Defectos por Entidad', fontweight='bold')
    axes[1, 1].set_ylabel('Cantidad')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 2. VALIDACIÓN DE DOMINIOS (VEHÍCULOS)

In [ ]:
vehiculos = tables.get('vehiculo', pd.DataFrame())

if len(vehiculos) > 0:
    print("🔍 VALIDACIÓN DE DOMINIOS")
    print(f"\nTotal vehículos: {len(vehiculos)}")
    
    # Dominios duplicados
    dominios_unicos = vehiculos['Dominio'].nunique()
    dominios_duplicados = len(vehiculos) - dominios_unicos
    
    print(f"Dominios únicos: {dominios_unicos}")
    print(f"Dominios duplicados: {dominios_duplicados}")
    print(f"Tasa de duplicación: {100*dominios_duplicados/len(vehiculos):.2f}%")
    
    # Top dominios duplicados
    dup_count = vehiculos['Dominio'].value_counts()
    dup_count = dup_count[dup_count > 1]
    
    if len(dup_count) > 0:
        print(f"\nTop 10 dominios con duplicados:")
        for dom, cnt in dup_count.head(10).items():
            print(f"  {dom:20s} {cnt:3d} vehículos")
    
    # Formatos inconsistentes
    print(f"\n📊 Análisis de formato:")
    print(f"  Largo promedio: {vehiculos['Dominio'].str.len().mean():.1f}")
    print(f"  Largo mín/máx: {vehiculos['Dominio'].str.len().min()}/{vehiculos['Dominio'].str.len().max()}")
    
    # Dominios con caracteres inválidos
    import re
    valid_pattern = r'^[A-Z]{2}\d{3}[A-Z]{2}$'
    invalid = vehiculos[~vehiculos['Dominio'].str.match(valid_pattern, na=False)]
    print(f"  Dominios con formato inválido: {len(invalid)} ({100*len(invalid)/len(vehiculos):.1f}%)")

## 3. CRUCE FLOTA-TELEMETRÍA

In [ ]:
dispositivos = tables.get('dispositivo', pd.DataFrame())
telemetria = tables.get('evento_telemetria', pd.DataFrame())

if len(vehiculos) > 0 and len(dispositivos) > 0:
    print("🔗 CRUCE FLOTA-TELEMETRÍA")
    print(f"\nVehículos totales: {len(vehiculos)}")
    print(f"Dispositivos registrados: {len(dispositivos)}")
    print(f"Eventos de telemetría: {len(telemetria)}")
    
    # Cobertura
    veh_con_disp = 0
    for placa in dispositivos['Placa'].unique():
        if placa in vehiculos['Dominio'].values:
            veh_con_disp += 1
    
    veh_sin_disp = len(vehiculos) - veh_con_disp
    
    print(f"\nCobertura de dispositivos:")
    print(f"  Con dispositivo: {veh_con_disp} ({100*veh_con_disp/len(vehiculos):.1f}%)")
    print(f"  Sin dispositivo: {veh_sin_disp} ({100*veh_sin_disp/len(vehiculos):.1f}%)")
    
    # Dispositivos por grupo
    print(f"\nDispositivos por grupo:")
    for grupo, count in dispositivos['Grupo'].value_counts().items():
        print(f"  {grupo:20s} {count:3d}")
    
    # Eventos por dispositivo
    if len(telemetria) > 0:
        eventos_por_disp = telemetria.groupby('dispositivo_id').size()
        print(f"\nEventos por dispositivo:")
        print(f"  Promedio: {eventos_por_disp.mean():.1f}")
        print(f"  Mediana: {eventos_por_disp.median():.1f}")
        print(f"  Rango: [{eventos_por_disp.min()}, {eventos_por_disp.max()}]")

In [ ]:
# Visualizar cobertura
if len(vehiculos) > 0 and len(dispositivos) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Análisis Flota-Telemetría', fontsize=12, fontweight='bold')
    
    # Cobertura pie
    cobertura = [veh_con_disp, veh_sin_disp]
    axes[0].pie(cobertura, labels=['Con dispositivo', 'Sin dispositivo'], autopct='%1.1f%%',
                colors=['#4CAF50', '#FF9800'])
    axes[0].set_title('Cobertura de Dispositivos')
    
    # Grupos
    dispositivos['Grupo'].value_counts().plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FFC107', '#F44336'])
    axes[1].set_title('Dispositivos por Grupo')
    axes[1].set_ylabel('Cantidad')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 4. ANÁLISIS DE TRANSACCIONES DE COMBUSTIBLE

In [ ]:
combustible = tables.get('transaccion_combustible', pd.DataFrame())

if len(combustible) > 0:
    print("⛽ ANÁLISIS DE COMBUSTIBLE")
    print(f"\nTransacciones totales: {len(combustible)}")
    
    print(f"\nValores faltantes:")
    for col in ['litros', 'precio_unitario', 'importe_total']:
        if col in combustible.columns:
            missing = combustible[col].isna().sum()
            print(f"  {col:20s} {missing:5d} ({100*missing/len(combustible):.1f}%)")
    
    # Estaciones
    print(f"\nTransacciones por estación:")
    for estacion, count in combustible['estacion'].value_counts().items():
        print(f"  {estacion:15s} {count:3d}")
    
    # Estadísticas de litros
    print(f"\nEstadísticas de litros (válidos):")
    litros = pd.to_numeric(combustible['litros'], errors='coerce')
    print(f"  Media: {litros.mean():.2f} L")
    print(f"  Mediana: {litros.median():.2f} L")
    print(f"  Desv. Est.: {litros.std():.2f} L")
    print(f"  Rango: [{litros.min():.2f}, {litros.max():.2f}] L")

## 5. ESQUEMA Y CALIDAD DE DATOS

In [ ]:
print("📋 ESQUEMA DE DATOS")
print(f"\nTabla: VEHICULO ({len(vehiculos)} rows × {len(vehiculos.columns)} cols)")
print("Columnas:")
for i, (col, dtype) in enumerate(vehiculos.dtypes.items(), 1):
    null_pct = 100 * vehiculos[col].isna().sum() / len(vehiculos)
    marker = '⚠' if null_pct > 50 else ' '
    print(f"  {i:2d}. {marker} {col:25s} | {str(dtype):10s} | Null: {null_pct:5.1f}%")

In [ ]:
print("\nTabla: DISPOSITIVO ({} rows × {} cols)".format(len(dispositivos), len(dispositivos.columns)))
print("Columnas:")
for i, (col, dtype) in enumerate(dispositivos.dtypes.items(), 1):
    null_pct = 100 * dispositivos[col].isna().sum() / len(dispositivos)
    marker = '⚠' if null_pct > 50 else ' '
    print(f"  {i:2d}. {marker} {col:30s} | {str(dtype):10s} | Null: {null_pct:5.1f}%")

## 6. RESUMEN EJECUTIVO

In [ ]:
print("\n" + "="*80)
print("📊 RESUMEN EJECUTIVO DEL DATASET SINTÉTICO")
print("="*80)

print(f"\n📈 VOLUMEN:")
for name, df in tables.items():
    if name != 'ejecucion_dataset':
        print(f"  {name:30s} {len(df):8d} registros × {len(df.columns):2d} campos")

print(f"\n🔴 DEFECTOS:")
if len(gt) > 0:
    print(f"  Total inyectados: {len(gt)}")
    for sev in ['alta', 'media', 'baja']:
        count = (gt['severidad'] == sev).sum()
        print(f"  {sev:15s} severidad: {count:3d}")

print(f"\n✅ ESTADO:")
if 'manifest' in locals():
    print(f"  Generado: {manifest.get('generated_at', 'N/A')}")
    print(f"  Seed: {manifest.get('seed')}")
    print(f"  Escenario: {manifest.get('scenario')}")

print(f"\n🎯 COBERTURA:")
if len(dispositivos) > 0:
    print(f"  Vehículos con telemetría: {100*veh_con_disp/len(vehiculos):.1f}%")
    print(f"  Dispositivos promedio/vehículo: {len(dispositivos)/veh_con_disp:.2f}")
    if len(telemetria) > 0:
        print(f"  Eventos promedio/dispositivo: {len(telemetria)/len(dispositivos):.1f}")

print("\n" + "="*80)